In [1]:
!pip install -qU torch torchvision torchaudio torchao trl peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9

In [2]:
%%writefile train.py


import os
import sys
import gc
import json
import random
import re
import subprocess
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback

# ============================================================
# Optional dependency bootstrap
# ============================================================

def ensure_dependencies() -> None:
    required = ["trl", "peft", "accelerate"]
    missing = []
    for pkg in required:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *missing])


ensure_dependencies()

try:
    from peft import LoraConfig, PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

try:
    from trl import GRPOConfig, GRPOTrainer
    HAS_TRL = True
except Exception:
    HAS_TRL = False

# ============================================================
# Environment / config
# ============================================================

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/hierarchical_steering_grpo_phi3"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

SEED = 42

# Keep these conservative for Kaggle 2xT4
N_GSM8K_TRAIN = 500
N_STRATEGYQA_TRAIN = 500
N_MMLU_TRAIN_PER_SUBJECT = 80

N_GSM8K_EVAL = 60
N_STRATEGYQA_EVAL = 60
N_MMLU_EVAL_PER_SUBJECT = 24

MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
    "formal_logic",
    "high_school_mathematics",
]

MAX_PROMPT_TOKENS = 768
MAX_COMPLETION_TOKENS = 128
NUM_GENERATIONS = 2
NUM_GENERATIONS_EVAL = 1
TEMPERATURE = 0.9
TOP_P = 0.95
LEARNING_RATE = 5e-6
MAX_STEPS = 250
EVAL_STEPS = 50
SAVE_STEPS = 50
LOGGING_STEPS = 10
GRAD_ACCUM_STEPS = 8
PER_DEVICE_BS = 1

NUM_ITERATIONS = 1
BETA = 0.02
LOSS_TYPE = "dr_grpo"
USE_VLLM = False
USE_LIGER_KERNEL = False
USE_PEFT = True

RUN_FINAL_EVAL = True

ROUTES = ["direct", "reason", "selfcheck"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [OUT_DIR, CSV_DIR, PLOTS_DIR, REPORTS_DIR, CHECKPOINT_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 180,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "font.family": "DejaVu Sans",
    }
)

# ============================================================
# Utilities
# ============================================================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def save_df(df: pd.DataFrame, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)


def _json_default(x):
    if isinstance(x, (np.integer, np.floating)):
        return x.item()
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, Path):
        return str(x)
    raise TypeError(f"Object of type {type(x).__name__} is not JSON serializable")

def save_json(obj: Any, path: str) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)


def free_memory(*objs) -> None:
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass


def preview_text(s: str, max_len: int = 120) -> str:
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")


def completion_text(c: Any) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, dict):
        return str(c.get("content", c))
    if isinstance(c, list) and c:
        first = c[0]
        if isinstance(first, dict):
            return str(first.get("content", first))
        return str(first)
    return str(c)


def sample_dataset(ds, n: int, seed: int):
    n = min(n, len(ds))
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(ds), size=n, replace=False).tolist()
    return ds.select(idx), idx


def annotated_bar(ax, fmt="{:.3f}") -> None:
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h):
            ax.annotate(
                fmt.format(h),
                (p.get_x() + p.get_width() / 2.0, h),
                ha="center",
                va="bottom",
                fontsize=8,
                xytext=(0, 3),
                textcoords="offset points",
            )


# ============================================================
# Parsers
# ============================================================

def extract_route(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<route>\s*(direct|reason|selfcheck)\s*</route>", text, flags=re.IGNORECASE)
    if m:
        return m[-1].lower()
    m = re.findall(r"\b(direct|reason|selfcheck)\b", text, flags=re.IGNORECASE)
    return m[-1].lower() if m else None


def extract_number(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = span.replace(",", "")

    boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
    if boxed:
        nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
        if nums:
            return nums[-1]

    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]

    hash_match = re.findall(r"####\s*(-?\d+\.?\d*)", text.replace(",", ""))
    if hash_match:
        return hash_match[-1]
    return None


def extract_yes_no(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text

    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()

    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None


def extract_mcq(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span_up = span.upper().strip()

    if span_up in ["A", "B", "C", "D"]:
        return span_up

    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"<ANSWER>\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()

    hits = re.findall(r"\b([ABCD])\b", span_up)
    return hits[-1].upper() if hits else None


def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None:
        return None
    try:
        x = float(str(pred).replace(",", ""))
        if abs(x - round(x)) < 1e-8:
            return str(int(round(x)))
        return str(x)
    except Exception:
        return None


def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try:
        return abs(float(pred) - float(gold)) <= 1e-6
    except Exception:
        return False


def canonical_letter_from_perm_answer(pred_letter: Optional[str], perm: Tuple[int, int, int, int]) -> Optional[str]:
    if pred_letter is None:
        return None
    pred_letter = pred_letter.upper().strip()
    if pred_letter not in ["A", "B", "C", "D"]:
        return None
    idx = ord(pred_letter) - 65
    return chr(65 + perm[idx])


def explicit_answer_present(task: str, text: str) -> bool:
    t = str(text)
    if task == "gsm8k":
        return bool(re.search(r"(\\boxed\{[-\d\.]+\}|-?\d+\.?\d*)", t))
    if task == "strategyqa":
        return bool(re.search(r"\b(yes|no)\b", t, flags=re.IGNORECASE))
    if task == "mmlu":
        return bool(re.search(r"\b([ABCD])\b", t, flags=re.IGNORECASE))
    return False


def answer_hedged(text: str) -> bool:
    t = str(text).lower()
    return any(h in t for h in ["maybe", "probably", "perhaps", "not sure", "uncertain", "depends"])


def difficulty_score(task: str, question: str) -> float:
    q = str(question).strip()
    if task == "gsm8k":
        ops = sum(q.count(x) for x in ["+", "-", "*", "/", "(", ")"])
        nums = len(re.findall(r"\d+", q))
        return min(1.0, 0.08 * len(q) / 10 + 0.12 * ops + 0.08 * nums)
    if task == "strategyqa":
        markers = sum(1 for w in [" not ", " never ", " except ", " least ", " most ", " all ", " any "] if w in f" {q.lower()} ")
        return min(1.0, 0.07 * len(q) / 10 + 0.18 * markers)
    if task == "mmlu":
        return min(1.0, 0.04 * len(q) / 10)
    return 0.5


def preferred_route(task: str, question: str) -> str:
    d = difficulty_score(task, question)
    if task == "gsm8k":
        return "selfcheck" if d >= 0.55 else "reason"
    if task == "strategyqa":
        return "selfcheck" if d >= 0.50 else "direct"
    if task == "mmlu":
        return "direct"
    return "reason"


# ============================================================
# Plotting
# ============================================================

def plot_training_history(df: pd.DataFrame, path: str) -> None:
    if len(df) == 0:
        return
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
    panels = [
        (axes[0, 0], [c for c in ["loss", "train_loss"] if c in df.columns], "Loss"),
        (axes[0, 1], [c for c in ["reward", "train_reward", "mean_reward"] if c in df.columns], "Reward"),
        (axes[1, 0], [c for c in ["eval_accuracy", "eval_reward", "eval_pass2_accuracy"] if c in df.columns], "Eval performance"),
        (axes[1, 1], [c for c in ["route_reward", "format_reward", "accuracy_reward", "length_reward", "hedge_reward"] if c in df.columns], "Reward terms"),
    ]
    for ax, cols, title in panels:
        for c in cols:
            ax.plot(df["step"], df[c], marker="o", linewidth=1.8, label=c)
        ax.set_title(title)
        ax.set_xlabel("Step")
        if cols:
            ax.legend(frameon=True, fontsize=8)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_eval_bars(summary_df: pd.DataFrame, path: str) -> None:
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    ax.bar(summary_df["task"], summary_df["accuracy"])
    ax.set_title("Task benchmark accuracy")
    ax.set_ylabel("Accuracy")
    annotated_bar(ax)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_route_distribution(df: pd.DataFrame, path: str) -> None:
    if len(df) == 0:
        return
    tab = pd.crosstab(df["task"], df["route_pred"].fillna("missing"), normalize="index")
    tab = tab.reindex(columns=ROUTES).fillna(0.0)
    heatmap = tab.values
    plt.figure(figsize=(8.8, 4.8))
    im = plt.imshow(heatmap, aspect="auto", vmin=0, vmax=max(0.1, float(heatmap.max())))
    plt.colorbar(im)
    plt.xticks(range(len(tab.columns)), tab.columns)
    plt.yticks(range(len(tab.index)), tab.index)
    plt.title("Route distribution by task")
    for i in range(heatmap.shape[0]):
        for j in range(heatmap.shape[1]):
            plt.text(j, i, f"{heatmap[i, j]:.2f}", ha="center", va="center", color="white" if heatmap[i, j] > 0.45 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_length_hist(df: pd.DataFrame, path: str) -> None:
    if len(df) == 0:
        return
    plt.figure(figsize=(8.5, 4.8))
    for task, g in df.groupby("task"):
        plt.hist(g["completion_len"], bins=25, alpha=0.5, label=task)
    plt.title("Completion length distribution")
    plt.xlabel("Tokens")
    plt.ylabel("Count")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_calibration_curve(df: pd.DataFrame, path: str) -> None:
    if "confidence" not in df.columns:
        return
    plt.figure(figsize=(6.8, 5.8))
    for task, g in df.groupby("task"):
        if len(g) == 0:
            continue
        conf = np.asarray(g["confidence"].tolist(), dtype=np.float64)
        corr = np.asarray(g["correct"].astype(int).tolist(), dtype=np.float64)
        bins = np.linspace(0, 1, 11)
        ids = np.clip(np.digitize(conf, bins) - 1, 0, 9)
        xs, ys = [], []
        for b in range(10):
            m = ids == b
            if m.sum() > 0:
                xs.append(conf[m].mean())
                ys.append(corr[m].mean())
        if xs:
            plt.plot(xs, ys, marker="o", linewidth=2, label=task)
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", alpha=0.6)
    plt.xlabel("Confidence")
    plt.ylabel("Accuracy")
    plt.title("Calibration curve")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


# ============================================================
# Dataset loading
# ============================================================

def load_gsm8k_split(split: str, n: int, tag: str):
    ds = load_dataset("gsm8k", "main", split=split)
    ds, idx = sample_dataset(ds, n, SEED)
    rows = []
    for i, s in enumerate(ds):
        gold = s["answer"].split("####")[-1].strip()
        rows.append(
            {
                "task": "gsm8k",
                "subject": "gsm8k",
                "sample_id": f"gsm8k_{split}_{i}_{tag}",
                "source_index": idx[i],
                "question": s["question"],
                "gold": canonical_number(gold),
                "raw_gold": gold,
            }
        )
    return rows


def load_strategyqa_split(split: str, n: int, tag: str):
    ds = load_dataset("ChilleD/StrategyQA", split=split)
    ds, idx = sample_dataset(ds, n, SEED)
    rows = []
    for i, s in enumerate(ds):
        rows.append(
            {
                "task": "strategyqa",
                "subject": "strategyqa",
                "sample_id": f"strategyqa_{split}_{i}_{tag}",
                "source_index": idx[i],
                "question": s["question"],
                "gold": "yes" if bool(s["answer"]) else "no",
            }
        )
    return rows


def load_mmlu_subject_split(subject: str, preferred_splits: List[str]):
    for sp in preferred_splits:
        try:
            return load_dataset("cais/mmlu", subject, split=sp), sp
        except Exception:
            continue
    raise RuntimeError(f"Could not load any split for {subject}")


def load_mmlu_split(split_name_candidates: List[str], n_per_subject: int, tag_prefix: str):
    rows = []
    for subject in MMLU_SUBJECTS:
        ds, chosen_split = load_mmlu_subject_split(subject, split_name_candidates)
        ds, idx = sample_dataset(ds, n_per_subject, SEED)
        for i, s in enumerate(ds):
            rows.append(
                {
                    "task": "mmlu",
                    "subject": subject,
                    "sample_id": f"mmlu_{subject}_{chosen_split}_{i}_{tag_prefix}",
                    "source_index": idx[i],
                    "question": s["question"],
                    "choices": list(s["choices"]),
                    "gold": chr(65 + int(s["answer"])),
                    "mmlu_split": chosen_split,
                }
            )
    return rows


# ============================================================
# Prompts / augmentation
# ============================================================

def build_prompt(sample: Dict[str, Any]) -> str:
    task = sample["task"]
    q = sample["question"]
    if task == "gsm8k":
        return (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then solve the problem with the route you chose.\n"
            "Output exactly: <route>...</route><think>...</think><answer>...</answer>\n"
            "For GSM8K, put the final numeric answer inside <answer> and prefer \\boxed{answer}.\n\n"
            f"Question: {q}\n"
        )
    if task == "strategyqa":
        return (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then answer the yes/no question.\n"
            "Output exactly: <route>...</route><think>...</think><answer>yes/no</answer>\n\n"
            f"Question: {q}\n"
        )
    if task == "mmlu":
        choices = sample["choices"]
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(choices)])
        return (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then answer the multiple choice question.\n"
            "Output exactly: <route>...</route><think>...</think><answer>A/B/C/D</answer>\n\n"
            f"Question: {q}\n{opts}\n"
        )
    raise ValueError(task)


def augment_mmlu_sample(sample: Dict[str, Any], perm: Tuple[int, int, int, int]) -> Dict[str, Any]:
    choices = list(sample["choices"])
    perm_choices = [choices[i] for i in perm]
    gold_idx = ord(sample["gold"]) - 65
    gold_perm_letter = chr(65 + perm.index(gold_idx))
    aug = dict(sample)
    aug["choices"] = perm_choices
    aug["gold"] = gold_perm_letter
    aug["perm"] = perm
    aug["gold_canonical"] = sample["gold"]
    return aug


def preferred_mode_for_sample(sample: Dict[str, Any]) -> str:
    return preferred_route(sample["task"], sample["question"])


def build_train_eval_sets():
    train_rows = []
    train_rows.extend(load_gsm8k_split("train", N_GSM8K_TRAIN, "gsm8k_train"))
    train_rows.extend(load_strategyqa_split("train", N_STRATEGYQA_TRAIN, "strategyqa_train"))
    train_rows.extend(load_mmlu_split(["auxiliary_train", "dev", "validation", "train"], N_MMLU_TRAIN_PER_SUBJECT, "mmlu_train"))

    aug_train_rows = []
    rng = random.Random(SEED)
    for r in train_rows:
        if r["task"] != "mmlu":
            rr = dict(r)
            rr["preferred_mode"] = preferred_mode_for_sample(rr)
            rr["difficulty"] = difficulty_score(rr["task"], rr["question"])
            rr["prompt"] = build_prompt(rr)
            rr["ground_truth"] = rr["gold"]
            aug_train_rows.append(rr)
            continue

        from itertools import permutations
        perms = list(permutations([0, 1, 2, 3], 4))
        rng.shuffle(perms)
        for perm in perms[:3]:
            aug = augment_mmlu_sample(r, perm)
            aug["preferred_mode"] = preferred_mode_for_sample(aug)
            aug["difficulty"] = difficulty_score(aug["task"], aug["question"])
            aug["prompt"] = build_prompt(aug)
            aug["ground_truth"] = aug["gold"]
            aug_train_rows.append(aug)

    eval_rows = []
    eval_rows.extend(load_gsm8k_split("test", N_GSM8K_EVAL, "gsm8k_eval"))
    eval_rows.extend(load_strategyqa_split("test", N_STRATEGYQA_EVAL, "strategyqa_eval"))
    eval_rows.extend(load_mmlu_split(["test"], N_MMLU_EVAL_PER_SUBJECT, "mmlu_eval"))

    for r in eval_rows:
        r["preferred_mode"] = preferred_mode_for_sample(r)
        r["difficulty"] = difficulty_score(r["task"], r["question"])
        r["prompt"] = build_prompt(r)
        r["ground_truth"] = r["gold"]

    all_keys = set()
    for r in aug_train_rows + eval_rows:
        all_keys.update(r.keys())

    for r in aug_train_rows + eval_rows:
        for k in all_keys:
            if k not in r:
                r[k] = None

    train_ds = Dataset.from_list(aug_train_rows).shuffle(seed=SEED)
    eval_ds = Dataset.from_list(eval_rows).shuffle(seed=SEED)
    return train_ds, eval_ds


# ============================================================
# Tokenizer / model
# ============================================================

def find_lora_target_modules(model) -> List[str]:
    suffixes = [
        "qkv_proj",
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_up_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
    found = set()
    for name, _ in model.named_modules():
        for suf in suffixes:
            if name.endswith(suf):
                found.add(suf)
    return sorted(found) if found else ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model_with_optional_sft():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    )

    model.config.use_cache = False
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except Exception:
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass

    if os.path.isdir(SFT_PATH) and HAS_PEFT:
        try:
            print(f"Loading and merging SFT adapter from {SFT_PATH} ...")
            model = PeftModel.from_pretrained(model, SFT_PATH)
            model = model.merge_and_unload()
            model.config.use_cache = False
        except Exception as e:
            print(f"Warning: failed to load/merge SFT adapter: {e}")

    return model


# ============================================================
# Evaluation
# ============================================================

def generate(model, tokenizer, prompt: str, max_new_tokens: int, temperature: float = 0.0, top_p: float = 1.0):
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = inputs["input_ids"].shape[1]

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p
    else:
        gen_kwargs["do_sample"] = False

    out = model.generate(**inputs, **gen_kwargs)
    full_output = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion


def parse_prediction(sample: Dict[str, Any], completion: str, mmlu_perm: Optional[Tuple[int, int, int, int]] = None) -> Dict[str, Any]:
    task = sample["task"]
    route = extract_route(completion)
    if task == "gsm8k":
        pred = canonical_number(extract_number(completion))
        correct = is_number_correct(pred, sample["gold"])
        return {"route_pred": route, "prediction": pred, "correct": bool(correct)}
    if task == "strategyqa":
        pred = extract_yes_no(completion)
        correct = pred == sample["gold"]
        return {"route_pred": route, "prediction": pred, "correct": bool(correct)}
    if task == "mmlu":
        pred = extract_mcq(completion)
        if pred is None:
            return {"route_pred": route, "prediction": None, "correct": False}
        if mmlu_perm is not None:
            pred = canonical_letter_from_perm_answer(pred, mmlu_perm)
        correct = pred == sample["gold"]
        return {"route_pred": route, "prediction": pred, "correct": bool(correct)}
    raise ValueError(task)


def eval_one_sample(model, tokenizer, sample: Dict[str, Any]):
    prompt = build_prompt(sample)
    full_output, completion = generate(model, tokenizer, prompt, MAX_COMPLETION_TOKENS, temperature=0.0)
    parsed = parse_prediction(sample, completion)
    route = parsed["route_pred"]
    pred = parsed["prediction"]
    correct = parsed["correct"]
    hedged = answer_hedged(completion)
    return {
        **sample,
        "prompt": prompt,
        "full_output": full_output,
        "completion": completion,
        "route_pred": route,
        "prediction": pred,
        "correct": bool(correct),
        "hedged": bool(hedged),
        "completion_len": len(tokenizer.encode(completion, add_special_tokens=False)),
        "format_ok": bool(route is not None and pred is not None and explicit_answer_present(sample["task"], completion)),
    }


def eval_suite(model, tokenizer, eval_ds: Dataset):
    rows = []
    for i, sample in enumerate(eval_ds):
        print(f"Eval {i+1}/{len(eval_ds)} | {sample['task']} | {preview_text(sample['question'], 90)}")
        if sample["task"] == "mmlu":
            rng = random.Random(SEED + i)
            from itertools import permutations
            perm_list = list(permutations([0, 1, 2, 3], 4))
            rng.shuffle(perm_list)
            perms = perm_list[:3]
            perm_rows = []
            base = dict(sample)
            for perm in perms:
                permuted = augment_mmlu_sample(base, perm)
                permuted["preferred_mode"] = preferred_mode_for_sample(permuted)
                permuted["difficulty"] = difficulty_score(permuted["task"], permuted["question"])
                permuted["prompt"] = build_prompt(permuted)
                r = eval_one_sample(model, tokenizer, permuted)
                r["perm"] = json.dumps(list(perm))
                r["canonical_prediction"] = canonical_letter_from_perm_answer(r["prediction"], perm) if r["prediction"] else None
                r["canonical_correct"] = bool(r["canonical_prediction"] == base["gold"])
                perm_rows.append(r)
            canon_preds = [r["canonical_prediction"] for r in perm_rows if r["canonical_prediction"] in ["A", "B", "C", "D"]]
            maj = Counter(canon_preds).most_common(1)[0][0] if canon_preds else None
            out = perm_rows[0]
            out["mmlu_raw_accuracy"] = float(np.mean([r["canonical_correct"] for r in perm_rows]))
            out["mmlu_majority_prediction"] = maj
            out["mmlu_majority_correct"] = bool(maj == base["gold"])
            out["mmlu_vote_fraction"] = float(Counter(canon_preds)[maj] / max(1, len(canon_preds))) if maj else 0.0
            rows.append(out)
        else:
            rows.append(eval_one_sample(model, tokenizer, sample))
        free_memory()
    df = pd.DataFrame(rows)

    if "canonical_correct" not in df.columns:
        df["canonical_correct"] = df["correct"]
    if "mmlu_majority_correct" not in df.columns:
        df["mmlu_majority_correct"] = df["correct"]
    if "mmlu_raw_accuracy" not in df.columns:
        df["mmlu_raw_accuracy"] = df["correct"]
    return df


def summarize_eval(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for task, g in df.groupby("task"):
        base = {
            "task": task,
            "n": int(len(g)),
            "route_rate": float(g["route_pred"].notna().mean()),
            "format_rate": float(g["format_ok"].mean()),
            "hedge_rate": float(g["hedged"].mean()),
            "avg_len": float(g["completion_len"].mean()),
        }
        if task == "mmlu":
            base["accuracy"] = float(g["canonical_correct"].mean())
            base["majority_accuracy"] = float(g["mmlu_majority_correct"].mean())
        else:
            base["accuracy"] = float(g["correct"].mean())
        rows.append(base)
    return pd.DataFrame(rows)


# ============================================================
# Rewards
# ============================================================

def accuracy_reward(prompts=None, completions=None, task=None, ground_truth=None, gold=None, choices=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for i, comp in enumerate(comps):
        t = task[i] if task is not None else None
        gt = ground_truth[i] if ground_truth is not None else (gold[i] if gold is not None else None)
        if t == "gsm8k":
            pred = canonical_number(extract_number(comp))
            rewards.append(2.0 if is_number_correct(pred, gt) else -2.0)
        elif t == "strategyqa":
            pred = extract_yes_no(comp)
            rewards.append(2.0 if pred == gt else -2.0)
        elif t == "mmlu":
            pred = extract_mcq(comp)
            rewards.append(2.0 if pred == gt else -2.0)
        else:
            rewards.append(0.0)
    return rewards


def format_reward(prompts=None, completions=None, task=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for comp in comps:
        has_route = re.search(r"<route>\s*(direct|reason|selfcheck)\s*</route>", comp, flags=re.IGNORECASE)
        has_think = re.search(r"<think>.*?</think>", comp, flags=re.IGNORECASE | re.DOTALL)
        has_answer = re.search(r"<answer>.*?</answer>", comp, flags=re.IGNORECASE | re.DOTALL)
        rewards.append(1.0 if (has_route and has_think and has_answer) else -1.0)
    return rewards


def route_reward(prompts=None, completions=None, preferred_mode=None, task=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for comp, pref in zip(comps, preferred_mode):
        route = extract_route(comp)
        if route is None:
            rewards.append(-0.5)
        elif route == pref:
            rewards.append(1.0)
        else:
            rewards.append(-0.25)
    return rewards


def brevity_reward(prompts=None, completions=None, task=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for comp, t in zip(comps, task):
        n = len(comp.split())
        target = 180 if t == "gsm8k" else 70 if t == "strategyqa" else 55
        penalty = abs(n - target) / max(1, target)
        rewards.append(float(0.5 - penalty))
    return rewards


def hedge_penalty(prompts=None, completions=None, task=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for comp in comps:
        t = comp.lower()
        hedge = any(h in t for h in ["maybe", "probably", "perhaps", "not sure", "uncertain", "depends"])
        rewards.append(-0.75 if hedge else 0.1)
    return rewards


def directness_reward(prompts=None, completions=None, task=None, **kwargs):
    comps = [completion_text(c) for c in completions]
    rewards = []
    for comp, t in zip(comps, task):
        answer = re.findall(r"<answer>(.*?)</answer>", comp, flags=re.IGNORECASE | re.DOTALL)
        ans = answer[-1].strip() if answer else ""
        if t == "gsm8k":
            ok = bool(re.search(r"\\boxed\{[-\d\.]+\}", ans) or re.search(r"-?\d+\.?\d*", ans))
        elif t == "strategyqa":
            ok = ans.lower() in ["yes", "no"]
        elif t == "mmlu":
            ok = ans.upper().strip() in ["A", "B", "C", "D"]
        else:
            ok = False
        rewards.append(0.5 if ok else -0.25)
    return rewards


# ============================================================
# Callback
# ============================================================

class HistoryCallback(TrainerCallback):
    def __init__(self, out_dir: str):
        self.out_dir = out_dir
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        row = {"step": int(state.global_step)}
        for k, v in logs.items():
            if isinstance(v, (int, float, np.floating)):
                row[k] = float(v)
        self.records.append(row)
        save_df(pd.DataFrame(self.records), os.path.join(self.out_dir, "training_history.csv"))


# ============================================================
# Main
# ============================================================

def main():
    if not HAS_TRL:
        raise RuntimeError("TRL is required for this script.")
    if not torch.cuda.is_available():
        print("Warning: CUDA is not available. This script is tuned for Kaggle GPUs.")

    print("Loading tokenizer ...")
    tokenizer = load_tokenizer()

    print("Loading model ...")
    model = load_model_with_optional_sft()

    print("Building datasets ...")
    train_ds, eval_ds = build_train_eval_sets()
    save_df(train_ds.to_pandas(), os.path.join(CSV_DIR, "train_dataset_preview.csv"))
    save_df(eval_ds.to_pandas(), os.path.join(CSV_DIR, "eval_dataset_preview.csv"))

    target_modules = find_lora_target_modules(model)
    print(f"LoRA target modules: {target_modules}")

    lora_config = None
    if USE_PEFT and HAS_PEFT:
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=target_modules,
        )

    reward_weights = [1.0, 0.7, 0.5, 0.2, 0.2, 0.2]

    training_args = GRPOConfig(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=PER_DEVICE_BS,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        max_steps=MAX_STEPS,
        logging_steps=LOGGING_STEPS,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        disable_dropout=True,
        remove_unused_columns=False,
        num_generations=NUM_GENERATIONS,
        num_generations_eval=NUM_GENERATIONS_EVAL,
        max_completion_length=MAX_COMPLETION_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        beta=BETA,
        num_iterations=NUM_ITERATIONS,
        loss_type=LOSS_TYPE,
        use_vllm=USE_VLLM,
        use_liger_kernel=USE_LIGER_KERNEL,
        log_completions=True,
        num_completions_to_print=2,
        log_unique_prompts=True,
        average_tokens_across_devices=True,
        repetition_penalty=1.05,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        reward_weights=reward_weights,
        fp16=torch.cuda.is_available(),
        bf16=False,
        use_cache=False,
        scale_rewards=False,
    )

    rewards = [accuracy_reward, format_reward, route_reward, brevity_reward, hedge_penalty, directness_reward]

    print("Starting GRPO training ...")
    hist_cb = HistoryCallback(OUT_DIR)
    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        reward_funcs=rewards,
        processing_class=tokenizer,
        peft_config=lora_config,
        callbacks=[hist_cb],
    )

    train_result = trainer.train()
    save_json(train_result.metrics if hasattr(train_result, "metrics") else {}, os.path.join(REPORTS_DIR, "train_metrics.json"))

    final_model_dir = os.path.join(CHECKPOINT_DIR, "final_model")
    trainer.save_model(final_model_dir)
    try:
        tokenizer.save_pretrained(final_model_dir)
    except Exception:
        pass

    history_df = pd.DataFrame(hist_cb.records)
    if len(history_df) > 0:
        save_df(history_df, os.path.join(CSV_DIR, "training_history.csv"))
        plot_training_history(history_df, os.path.join(PLOTS_DIR, "training_history.png"))

    final_summary = pd.DataFrame()
    final_eval = pd.DataFrame()

    if RUN_FINAL_EVAL:
        print("Running final evaluation ...")
        final_model = trainer.model
        final_model.eval()

        final_eval = eval_suite(final_model, tokenizer, eval_ds)
        final_summary = summarize_eval(final_eval)
        save_df(final_eval, os.path.join(CSV_DIR, "final_eval_predictions.csv"))
        save_df(final_summary, os.path.join(CSV_DIR, "final_summary.csv"))

        plot_eval_bars(final_summary, os.path.join(PLOTS_DIR, "final_task_accuracy.png"))
        plot_route_distribution(final_eval, os.path.join(PLOTS_DIR, "final_route_distribution.png"))
        plot_length_hist(final_eval, os.path.join(PLOTS_DIR, "final_length_hist.png"))
        plot_calibration_curve(final_eval, os.path.join(PLOTS_DIR, "final_calibration.png"))

        mmlu_eval = final_eval[final_eval["task"] == "mmlu"].copy()
        if len(mmlu_eval) > 0:
            subj = mmlu_eval.groupby("subject", as_index=False).agg(
                n=("correct", "count"),
                raw_accuracy=("canonical_correct", "mean"),
                majority_accuracy=("mmlu_majority_correct", "mean"),
                route_rate=("route_pred", lambda s: float(s.notna().mean())),
                avg_len=("completion_len", "mean"),
            )
            save_df(subj, os.path.join(CSV_DIR, "mmlu_subject_summary.csv"))
            plt.figure(figsize=(10.5, 4.8))
            plt.bar(subj["subject"], subj["majority_accuracy"])
            plt.xticks(rotation=30, ha="right")
            plt.ylabel("Accuracy")
            plt.title("MMLU majority-vote accuracy by subject")
            annotated_bar(plt.gca())
            plt.tight_layout()
            plt.savefig(os.path.join(PLOTS_DIR, "mmlu_subject_accuracy.png"), dpi=180)
            plt.close()

    report_lines = []
    report_lines.append("# Hierarchical steering GRPO report")
    report_lines.append("")
    report_lines.append(f"- Base model: `{BASE_MODEL}`")
    report_lines.append(f"- Starting checkpoint: `{SFT_PATH if os.path.exists(SFT_PATH) else 'base model'}`")
    report_lines.append(f"- Training samples: {len(train_ds)}")
    report_lines.append(f"- Evaluation samples: {len(eval_ds)}")
    report_lines.append(f"- GRPO loss type: `{LOSS_TYPE}`")
    report_lines.append(f"- Number of generations per prompt: {NUM_GENERATIONS}")
    report_lines.append(f"- Max completion length: {MAX_COMPLETION_TOKENS}")
    report_lines.append("")
    report_lines.append("## Method notes")
    report_lines.append("- The model learns a route token first, then generates the answer in a constrained format.")
    report_lines.append("- The reward stack combines correctness, format adherence, route alignment, brevity, and anti-hedging.")
    report_lines.append("- MMLU training is augmented with random choice permutations to reduce answer-order sensitivity.")
    report_lines.append("- Training graphs, CSVs, and saved checkpoints are written to the Kaggle working directory.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(report_lines))

    save_json(
        {
            "config": {
                "BASE_MODEL": BASE_MODEL,
                "SFT_PATH": SFT_PATH,
                "SEED": SEED,
                "N_GSM8K_TRAIN": N_GSM8K_TRAIN,
                "N_STRATEGYQA_TRAIN": N_STRATEGYQA_TRAIN,
                "N_MMLU_TRAIN_PER_SUBJECT": N_MMLU_TRAIN_PER_SUBJECT,
                "N_GSM8K_EVAL": N_GSM8K_EVAL,
                "N_STRATEGYQA_EVAL": N_STRATEGYQA_EVAL,
                "N_MMLU_EVAL_PER_SUBJECT": N_MMLU_EVAL_PER_SUBJECT,
                "MMLU_SUBJECTS": MMLU_SUBJECTS,
                "MAX_COMPLETION_TOKENS": MAX_COMPLETION_TOKENS,
                "NUM_GENERATIONS": NUM_GENERATIONS,
                "LEARNING_RATE": LEARNING_RATE,
                "MAX_STEPS": MAX_STEPS,
                "LOSS_TYPE": LOSS_TYPE,
            },
            "final_summary": final_summary.to_dict(orient="records") if len(final_summary) else [],
        },
        os.path.join(REPORTS_DIR, "summary.json"),
    )

    print("\nSaved outputs to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- Reports: {REPORTS_DIR}")
    print(f"- Checkpoints: {CHECKPOINT_DIR}")

    free_memory(model)


if __name__ == "__main__":
    main()


Writing train.py


In [3]:
!accelerate launch --num_processes=2 train.py

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
W0617 11:32:49.093000 93 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0617 11:32:49.111000 94 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future re